# Fib (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_cpp.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `Fib`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono variabili globali, e la traccia si attiva con `trace = true`.

## Numeri di Fibonacci: Fib

**Input:** un numero naturale $n$. **Output:** $F(n)$, con $F(0) = 0$, $F(1) = 1$ e $F(n) = F(n-1) + F(n-2)$. Algoritmo ricorsivo naive: $C(n) = 2F(n+1) - 1$ chiamate e $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ passi.

Pseudocodice: sezione 14.5 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [1]:
%%writefile fib.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
long long calls = 0;    // numero di chiamate = nodi dell'albero delle chiamate
int maxDepth = 0;       // profondita' della ricorsione = altezza massima dello stack
long long mults = 0;    // moltiplicazioni (FastPower)
long long moves = 0;    // mosse (Hanoi)
int ev = 0;             // contatore degli eventi della traccia
int stk[1000];          // lo stack delle chiamate: gli argomenti delle chiamate attive (al massimo 1000)
int sp = 0;             // numero di chiamate attive
long long seen[64];     // seen[x] = numero di chiamate con argomento x
bool trace = false;     // true: stampa la traccia dell'esecuzione

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Riga | Esecuzioni\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Passi totali S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // come sprintf, ma restituisce una stringa
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // la chiamata termina: il suo frame esce dallo stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // riga numerata della traccia, con lo stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

// Fib(n)
long long Fib(int n)
{
    Push(n); cnt[2]++;
    if (n <= 1)
    {
        cnt[3]++;
        if (trace) Say(Str("Fib(%d)", n), Str("caso base, restituisce %d", n));
        Pop();
        return n;
    }
    if (trace) Say(Str("Fib(%d)", n), Str("chiama Fib(%d)", n - 1));
    long long x = Fib(n - 1);                      // le due chiamate sono separate solo per stampare la traccia
    if (trace) Say(Str("Fib(%d)", n), Str("chiama Fib(%d)", n - 2));
    long long y = Fib(n - 2);
    cnt[5]++;
    if (trace) Say(Str("Fib(%d)", n), Str("restituisce %lld + %lld = %lld", x, y, x + y));
    Pop();
    return x + y;
}

long long FibValue(int k)                          // F(k) con un ciclo, per controllare
{
    long long a = 0, b = 1;
    for (int i = 0; i < k; i++)
    {
        long long t = a + b;
        a = b;
        b = t;
    }
    return a;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Esempio delle dispense: l'esecuzione di Fib(4), in ordine di visita in profondita'
    int n = 4;
    Reset();
    trace = true;
    printf("%-28s| %s\n", "chiamata", "azione");
    long long r = Fib(n);
    trace = false;
    printf("F(%d) = %lld, chiamate C(%d) = %lld = 2F(%d) - 1, profondita' = %d\n", n, r, n, calls, n + 1, maxDepth);

    // Fib(5): gli stessi sottoproblemi sono risolti piu' volte
    printf("\n");
    n = 5;
    Reset();
    r = Fib(n);
    printf("Fib(%d) = %lld: %lld chiamate, profondita' %d, S = %lld passi\n", n, r, calls, maxDepth, Total());
    printf("  k | chiamate di Fib(k)\n");
    for (int k = n; k >= 0; k--) printf("%3d | %lld\n", k, seen[k]);
    PrintSteps({2, 3, 5});

    // Numero di chiamate e di passi: C(n) = 2F(n+1) - 1 e S(n) = 4F(n+1) - 2
    printf("\n");
    printf("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | profondita'\n");
    int values[] = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30};
    for (int v : values)
    {
        Reset();
        r = Fib(v);
        printf("%3d | %6lld | %8lld | %11lld | %8lld | %11lld | %d\n", v, r, calls, 2 * FibValue(v + 1) - 1, Total(), 4 * FibValue(v + 1) - 2, maxDepth);
    }
    return 0;
}
#endif

Writing fib.cpp


In [2]:
!g++ -std=c++17 -O2 -o fib fib.cpp && ./fib

chiamata                    | azione
Fib(4)                      | chiama Fib(3)
  Fib(3)                    | chiama Fib(2)
    Fib(2)                  | chiama Fib(1)
      Fib(1)                | caso base, restituisce 1
    Fib(2)                  | chiama Fib(0)
      Fib(0)                | caso base, restituisce 0
    Fib(2)                  | restituisce 1 + 0 = 1
  Fib(3)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(3)                    | restituisce 1 + 1 = 2
Fib(4)                      | chiama Fib(2)
  Fib(2)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(2)                    | chiama Fib(0)
    Fib(0)                  | caso base, restituisce 0
  Fib(2)                    | restituisce 1 + 0 = 1
Fib(4)                      | restituisce 2 + 1 = 3
F(4) = 3, chiamate C(4) = 9 = 2F(5) - 1, profondita' = 4

Fib(5) = 5: 15 chiamate, profondita' 5, S = 30 passi
  k | chiamate di

**Prova tu.** Cambia $n$ e guarda quanto crescono le chiamate.

In [3]:
%%writefile prova_fib.cpp
#define SOLO_LIBRERIA   // non eseguire il main di fib.cpp
#include "fib.cpp"

int main()
{
    int n = 6;        // <-- cambia n (con n > 45 ci vuole un po')
    Reset();
    long long r = Fib(n);
    printf("F(%d) = %lld, chiamate = %lld, S = %lld\n", n, r, calls, Total());
    return 0;
}

Writing prova_fib.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_fib prova_fib.cpp && ./prova_fib

F(6) = 8, chiamate = 25, S = 50


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).